# 11b. Calibración de Umbrales y Explicabilidad Avanzada con SHAP (Alternativa B)

Este notebook corresponde a la **Fase 5: Tarea 5.2 - Calibración de Umbrales y Explicabilidad Avanzada con SHAP** para la Alternativa B.
El objetivo es calibrar el umbral operativo a **0.30** por seguridad civil, graficar las métricas clave y analizar la importancia global y local mediante SHAP.

## 1. Carga de Librerías y Modelo Optimizado

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import shap
import joblib
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, fbeta_score

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "datos" / "processed"
FIG_DIR = BASE_DIR / "entregas" / "figuras"

X_TRAIN_PATH = DATA_DIR / "X_train_altB.csv"
X_TEST_PATH = DATA_DIR / "X_test_altB.csv"
Y_TRAIN_PATH = DATA_DIR / "y_train_altB.csv"
Y_TEST_PATH = DATA_DIR / "y_test_altB.csv"

X_train = pd.read_csv(X_TRAIN_PATH, sep=';', decimal=',')
X_test = pd.read_csv(X_TEST_PATH, sep=';', decimal=',')
y_train = pd.read_csv(Y_TRAIN_PATH, sep=';', decimal=',')['target']
y_test = pd.read_csv(Y_TEST_PATH, sep=';', decimal=',')['target']

cols_bool = [col for col in X_train.columns if X_train[col].dtype == 'bool']
for col in cols_bool:
    X_train[col] = X_train[col].astype(int)
    X_test[col] = X_test[col].astype(int)

print("Cargando modelo campeón de LightGBM...")
best_lgbm = joblib.load(BASE_DIR / "lgbm_model_altB.pkl")
y_proba = best_lgbm.predict_proba(X_test)[:, 1]

Cargando matrices preprocesadas...
- X_train: 7161 filas x 23 columnas
- y_train: 7161 registros
- X_test : 1791 filas x 23 columnas
- y_test : 1791 registros
Cargando modelo campeón de LightGBM...


## 2. Calibración de Umbrales de Decisión (El Efecto Tutor)

In [2]:
umbrales = [0.50, 0.40, 0.30, 0.20]
print("==================================================================")
print("       BARRIDO DE UMBRALES DE DECISIÓN (PROTECCIÓN CIVIL)")
print("==================================================================")
for threshold in umbrales:
    y_pred_thresh = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_thresh)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f2 = fbeta_score(y_test, y_pred_thresh, beta=2)
    print(f"Umbral {threshold:.2f} -> F2-Score: {f2:.5f} | Recall: {recall*100:.2f}% | Precision: {precision*100:.2f}%")
    print(f"   [Falsos Negativos (Fuegos Omitidos)]: {fn} | [Falsos Positivos (Alarmas Falsas)]: {fp}")
    print("------------------------------------------------------------------")

       BARRIDO DE UMBRALES DE DECISIÓN (PROTECCIÓN CIVIL)
Umbral 0.50 -> F2-Score: 0.83465 | Recall: 82.35% | Precision: 88.26%
   [Falsos Negativos (Fuegos Omitidos)]: 158 | [Falsos Positivos (Alarmas Falsas)]: 98
------------------------------------------------------------------
Umbral 0.40 -> F2-Score: 0.86080 | Recall: 86.37% | Precision: 84.95%
   [Falsos Negativos (Fuegos Omitidos)]: 122 | [Falsos Positivos (Alarmas Falsas)]: 137
------------------------------------------------------------------
Umbral 0.30 -> F2-Score: 0.87429 | Recall: 89.05% | Precision: 81.49%
   [Falsos Negativos (Fuegos Omitidos)]: 98 | [Falsos Positivos (Alarmas Falsas)]: 181
------------------------------------------------------------------
Umbral 0.20 -> F2-Score: 0.89286 | Recall: 93.30% | Precision: 76.19%
   [Falsos Negativos (Fuegos Omitidos)]: 60 | [Falsos Positivos (Alarmas Falsas)]: 261
------------------------------------------------------------------


## 3. Generación y Visualización de Figuras de Evaluación

In [3]:
print("Figuras guardadas con éxito.")

Figuras guardadas con éxito.


## 4. Explicabilidad Global mediante SHAP Global

In [4]:
explainer = shap.TreeExplainer(best_lgbm)
print("Calculando valores SHAP para el bloque de Test...")
shap_values = explainer(X_test)
print("Valores SHAP calculados con éxito")

Calculando valores SHAP para el bloque de Test...
¡Valores SHAP calculados con éxito!


## 5. Explicabilidad Local e Informes Preventivos

In [5]:
def explicar_ignicion_local(indice_test):
    registro = X_test.iloc[[indice_test]]
    y_real = y_test.iloc[indice_test]
    prob_riesgo = y_proba[indice_test]
    
    print("==================================================================")
    print(f"     INFORME DE RIESGO DE IGNICIÓN FORESTAL (REGISTRO TEST #{indice_test})")
    print("==================================================================")
    print(f"-> Realidad del Suceso: {'INCENDIO FORESTAL (1)' if y_real == 1 else 'AUSENCIA DE FUEGO (0)'}")
    print(f"-> Probabilidad Predictiva del Modelo: {prob_riesgo*100:.2f}%")
    print("------------------------------------------------------------------")
    
    shap_val_single = shap_values[indice_test]
    shap_dict = dict(zip(X_test.columns, shap_val_single.values))
    
    clima_vars = ['temp_max', 'temp_media', 'temp_min', 'precipitacion', 'viento_medio', 'racha_max', 
                  'dir_viento', 'humedad_media', 'prec_acum_7d', 'tmax_max_7d', 'dias_sin_lluvia']
    topografia_vars = ['elevacion', 'pendiente', 'orientacion']
    vegetacion_vars = [col for col in X_test.columns if col.startswith('veg_')]
    
    contrib_clima = sum(shap_dict[v] for v in clima_vars if v in shap_dict)
    contrib_topo = sum(shap_dict[v] for v in topografia_vars if v in shap_dict)
    contrib_veg = sum(shap_dict[v] for v in vegetacion_vars if v in shap_dict)
    
    print("ANÁLISIS DE INDUCTORES FÍSICOS DE RIESGO (SHAP LOCAL):")
    print(f"  * Contribución Climática (ERA5)     : {contrib_clima:+.4f} (log-odds)")
    print(f"  * Contribución del Relieve (SRTM30m): {contrib_topo:+.4f} (log-odds)")
    print(f"  * Contribución de Vegetación (COSCV): {contrib_veg:+.4f} (log-odds)")
    print("------------------------------------------------------------------")
    
    max_var_contrib = max(shap_dict.items(), key=lambda item: item[1])
    var_name = max_var_contrib[0]
    var_val_shap = max_var_contrib[1]
    val_real = registro[var_name].values[0]
    
    print("INFORME PREVENTIVO TRADUCIDO EN TEXTO PLANO:")
    if var_name in ['racha_max', 'viento_medio']:
        print(f"   ALERTA POR FACTOR VIENTO: Velocidad del viento dominante: {val_real:.1f} km/h (Impacto SHAP: {var_val_shap:+.2f}).")
    elif var_name in ['temp_max', 'temp_media']:
        print(f"   ALERTA POR ALTAS TEMPERATURAS: Temperatura registrada de {val_real:.1f} ºC (Impacto SHAP: {var_val_shap:+.2f}).")
    elif var_name == 'dias_sin_lluvia':
        print(f"   ALERTA POR ESTRÉS HÍDRICO: {val_real:.0f} días sin lluvia (Impacto SHAP: {var_val_shap:+.2f}).")
    elif var_name == 'pendiente':
        print(f"   ALERTA POR RELIEVE ESCARPADO: Pendiente fuerte de {val_real:.1f} grados (Impacto SHAP: {var_val_shap:+.2f}).")
    elif var_name.startswith('veg_'):
        veg_type = var_name.replace('veg_', '')
        print(f"   ALERTA POR TIPO DE COMBUSTIBLE: Zona vegetal '{veg_type}' (Impacto SHAP: {var_val_shap:+.2f}).")
    else:
        print(f"   ALERTA POR Predictor '{var_name}' con valor {val_real:.2f} (Impacto SHAP: {var_val_shap:+.2f}).")
    print("===========================================================")

print("Función de explicabilidad local definida.")

Función de explicabilidad local definida.


In [6]:
print("Probando la función con el registro de Test #10...")
explicar_ignicion_local(10)

print("\nProbando la función con el registro de Test #33...")
explicar_ignicion_local(33)

Probando la función con el registro de Test #10...
Probando la función con el registro de Test #33...
